<a href="https://colab.research.google.com/github/Anish19s/ASR_Pipeline/blob/main/Copy_of_asrpipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALLING LIBRARIES

In [ ]:
pip install transformers datasets torchaudio evaluate jiwer librosa sounddevice scipy soundfile

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sounddevice-0.5.5-py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.8 MB/s eta 0:00:00


DELETE ALL LOCALLY STORED DOWNLOADS AND CACHED FILES

In [ ]:
!rm -rf ~/.cache/huggingface
!rm -rf ~/.cache/datasets

UPLOADING AUDIO FILE(OPTION 1)

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving test3.ogg to test3.ogg


PREPROCESSING AUDIO

In [ ]:
import torchaudio
import torchaudio.transforms as T

waveform,sampling_rate=torchaudio.load("test3.ogg")

print(waveform.shape)
print(sampling_rate)

if waveform.shape[0]>1:
    waveform=waveform.mean(dim=0)
else:
  waveform=waveform.squeeze()


target_rate=16000

if sampling_rate!=target_rate:
  resampler=T.Resample(sampling_rate,target_rate)
  waveform=resampler(waveform)
  sampling_rate=target_rate


waveform=waveform/waveform.abs().max()

print(waveform.shape)
print(waveform.abs().mean())
print(waveform.max(), waveform.min())


print(waveform.shape)
print(sampling_rate)

torch.Size([1, 2070616])
48000
torch.Size([690206])
tensor(0.0415)
tensor(0.9927) tensor(-1.)
torch.Size([690206])
16000


TAKING USER INPUT(OPTION 2)

In [ ]:
!pip install gradio

In [ ]:
from transformers import WhisperProcessor,WhisperForConditionalGeneration
import torch

processor=WhisperProcessor.from_pretrained("openai/whisper-small")
model=WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


device="cuda" if torch.cuda.is_available() else "cpu"
model=model.to(device)



def transcribe(audio):
    waveform,sampling_rate=audio


    if waveform.shape[0]>1:
        waveform=waveform.mean(dim=0)
    else:
      waveform=waveform.squeeze()


    target_rate=16000

    if sampling_rate!=target_rate:
      resampler=T.Resample(sampling_rate,target_rate)
      waveform=resampler(waveform)
      sampling_rate=target_rate


    waveform=waveform/waveform.abs().max()


    chunk_len=30*sampling_rate
    stride=5*sampling_rate
    chunks=[]

    for i in range(0,waveform.shape[0],chunk_len-stride):
        chunk=waveform[i:i+chunk_len]
        chunks.append(chunk)


    final_text=""

    for chunk in chunks:

        inputs=processor(chunk,sampling_rate=16000,return_tensors="pt",return_attention_mask=True)
        input_features=inputs["input_features"].to(device)


        with torch.no_grad():
            predicted_ids=model.generate(input_features)


        transcription=processor.batch_decode(predicted_ids,skip_special_tokens=True)[0]

        final_text+=transcription+" "


    return final_text


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(sources="microphone", type="numpy"),
    outputs="text",
    title="🎤 My ASR Speech to Text Model"
)

demo.launch(share=False,debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

USING A KNOWN DATASET(OPTION 3)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    split="validation"
)


sample=dataset[1]
reference=sample["text"]
print(sample["text"])

audio=sample["audio"]

waveform=audio["array"]
sampling_rate=audio["sampling_rate"]

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER


LISTEN TO AUDIO

In [ ]:
from IPython.display import Audio

Audio(waveform,rate=sampling_rate)

PROCESSING DATA IN CHUNKS

In [ ]:
from transformers import WhisperProcessor,WhisperForConditionalGeneration
import torch

processor=WhisperProcessor.from_pretrained("openai/whisper-small")
model=WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


inputs=processor(waveform,sampling_rate=16000,return_tensors="pt",return_attention_mask=True)

device="cuda" if torch.cuda.is_available() else "cpu"
model=model.to(device)


chunk_len=30*sampling_rate
stride=5*sampling_rate
chunks=[]

for i in range(0,waveform.shape[0],chunk_len-stride):
    chunk=waveform[i:i+chunk_len]
    chunks.append(chunk)


final_text=""

for chunk in chunks:

    inputs=processor(chunk,sampling_rate=16000,return_tensors="pt",return_attention_mask=True)
    input_features=inputs["input_features"].to(device)


    with torch.no_grad():
        predicted_ids=model.generate(input_features)


    transcription=processor.batch_decode(predicted_ids,skip_special_tokens=True)[0]

    final_text+=transcription


print(final_text)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

 For G block and F block you lock your room and while going for vacation Otherwise your furniture will be taken by someone else Lock you will be continuing the same Room itself, okay for the next semester and while going make sure that you are cleaning your Sunset area also with the stick you put all the waste in the sunset area to the ground you put all the waste in the sunset area to the ground and also clean your room while going you update the occupancy register in the office they will not down


EVALUATION

In [ ]:
from evaluate import load

import re

def normalize(text):
    text = text.lower()                 # lowercase
    text = re.sub(r"[^\w\s]", "", text) # remove punctuation
    return text
reference_clean = normalize(reference)
prediction_clean = normalize(transcription)

wer_metric=load("wer")
wer = wer_metric.compute(
    predictions=[prediction_clean],
    references=[reference_clean]
)


print(wer)

0.1
